Only to ensure Onyxia GPUs to work.

In [1]:
# Onyxia GPU compatible torch version
import sys
!{sys.executable} -m pip install torch==2.11.0 torchaudio torchvision \
    --index-url https://download.pytorch.org/whl/cu126 \
    --force-reinstall

Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 10.0 MB/s  0:00:40:00:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 16.4 MB/s  0:00:19:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 29.7 MB/s  0:00:10:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 28.0 MB/s  0:00:05ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 9.4 MB/s  0:00:310:00:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 4.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 63.8 MB/s  0:00:03ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 172.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 105.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 97.6 MB/s  0:00:00 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 98.2 MB/s  0:

In [2]:
import pandas as pd
import os
import numpy as np
import re
import string
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


*If you want to analyse another sport, please change the path accordingly.*

Load the corpus.

In [ ]:
data_path = "PESSD/biath/"

df = pd.DataFrame()
for file in os.listdir(data_path):
    if file.endswith(".csv"):
        temp = pd.read_csv(os.path.join(data_path, file))
        df = pd.concat([df, temp])

In [14]:
# Filter the data

df = df[(df["main_g"]=="female") | (df["main_g"]=="male")]
df["text"] = df["text"].apply(lambda x: x.split(". "))
df = df.explode("text")
df["text"] = df["text"].apply(lambda x: re.sub(r"\s*[A-Z]\w*\s*", " ", x).strip())
df["text"] = df["text"].apply(lambda x: x.translate(str.maketrans('', '', string.punctuation.replace("'", ""))))
df["ID"] = df["audio_file"].apply(lambda x: x.replace("_wav.wav", ""))
df = df.merge(pd.read_csv("metadata.csv"), on="ID")
df = df.reset_index(drop=True)


When preparing the data, it is necessary to analyse the corpus vocabulary to identify terms relevant to cosine similarity, ensuring that they are present in both the male and female corpora.

In [15]:
STOPWORDS = [x.strip() for x in open('stopwords.txt').readlines()]

words = " ".join(df["text"].tolist()).split()
words_filtered = [w for w in words if w not in STOPWORDS and len(w) > 2]
freq = Counter(words_filtered).most_common(50)

words_H = " ".join(df[df["g_ath"] == "H"]["text"].tolist()).split()
words_F = " ".join(df[df["g_ath"] == "F"]["text"].tolist()).split()

counter_H = Counter([w for w in words_H if w not in STOPWORDS and len(w) > 2])
counter_F = Counter([w for w in words_F if w not in STOPWORDS and len(w) > 2])

print("=== 50 mots les plus fréquents (hors stopwords) ===")
for word, count in freq:
    h = counter_H.get(word, 0)
    f = counter_F.get(word, 0)
    total = h + f
    pct_h = h / total * 100 if total > 0 else 0
    pct_f = f / total * 100 if total > 0 else 0
    print(f"  {word}: {count} (H: {pct_h:.0f}% | F: {pct_f:.0f}%)")
print(f"Total H : {len([w for w in words_H if w not in STOPWORDS and len(w) > 2])/len(words_filtered)} | F : {len([w for w in words_F if w not in STOPWORDS and len(w) > 2])/len(words_filtered)}")

=== 50 mots les plus fréquents (hors stopwords) ===
  'est: 682 (H: 63% | F: 37%)
  secondes: 671 (H: 55% | F: 45%)
  tir: 456 (H: 59% | F: 41%)
  course: 306 (H: 57% | F: 43%)
  relais: 277 (H: 70% | F: 30%)
  faut: 247 (H: 63% | F: 37%)
  temps: 246 (H: 61% | F: 39%)
  médaille: 237 (H: 66% | F: 34%)
  vraiment: 234 (H: 67% | F: 33%)
  tête: 230 (H: 58% | F: 42%)
  petit: 227 (H: 59% | F: 41%)
  déjà: 215 (H: 57% | F: 43%)
  skis: 190 (H: 57% | F: 43%)
  également: 183 (H: 48% | F: 52%)
  vite: 182 (H: 50% | F: 50%)
  français: 176 (H: 85% | F: 15%)
  piste: 173 (H: 60% | F: 40%)
  peutêtre: 169 (H: 62% | F: 38%)
  monde: 167 (H: 60% | F: 40%)
  l'instant: 155 (H: 66% | F: 34%)
  voit: 147 (H: 69% | F: 31%)
  chercher: 145 (H: 51% | F: 49%)
  qu'on: 145 (H: 64% | F: 36%)
  olympique: 143 (H: 63% | F: 37%)
  tour: 134 (H: 63% | F: 37%)
  aller: 133 (H: 54% | F: 46%)
  coucher: 125 (H: 49% | F: 51%)
  train: 117 (H: 63% | F: 37%)
  sprint: 116 (H: 51% | F: 49%)
  ski: 111 (H: 61% | F: 

Calculation of embeddings

In [19]:
embeddings_path = data_path+"embeddings.npy"

if os.path.exists(embeddings_path):
    print("\nChargement des embeddings depuis le cache...")
    embeddings = np.load(embeddings_path)
    print(f"Embeddings chargés : {embeddings.shape}")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nCalcul des embeddings sur : {device}")

    embedding_model = SentenceTransformer("dangvantuan/sentence-camembert-base", device=device)

    embeddings = embedding_model.encode(
        df["text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    np.save(embeddings_path, embeddings)
    print(f"Embeddings sauvegardés dans {embeddings_path}")

df["embedding"] = list(embeddings)
print(f"Corpus : {len(df)} phrases | embeddings : {embeddings.shape}")



Chargement des embeddings depuis le cache...
Embeddings chargés : (8844, 768)
Corpus : 8844 phrases | embeddings : (8844, 768)


When calculating the cosine similarity by term, the list of terms must be adjusted based on a selection of the most frequently occurring words, focusing on the most relevant terms. 

In [20]:
TERMES = ["tir", "course", "sprint", "olympique", "secondes", "coucher", "ski", "vraiment","médaille","fort","rapide"]

results = []

for terme in TERMES:
    mask_F = df["g_ath"].str.upper() == "F"
    mask_H = df["g_ath"].str.upper() == "H"
    mask_terme = df["text"].str.contains(terme, case=False, na=False)

    emb_F = np.array(df[mask_F & mask_terme]["embedding"].tolist())
    emb_H = np.array(df[mask_H & mask_terme]["embedding"].tolist())

    if len(emb_F) == 0 or len(emb_H) == 0:
        results.append({"terme": terme,  "similarite_cosinus": None})
        continue

    centroid_F = emb_F.mean(axis=0, keepdims=True)
    centroid_H = emb_H.mean(axis=0, keepdims=True)

    sim = cosine_similarity(centroid_F, centroid_H)[0][0]
    results.append({"terme": terme, "similarite_cosinus": round(sim, 4)})


results_df = pd.DataFrame(results).dropna(subset=["similarite_cosinus"]).sort_values("similarite_cosinus")
print("\n=== Similarité cosinus F vs H par terme ===")
print(results_df.to_string(index=False))


=== Similarité cosinus F vs H par terme ===
    terme  similarite_cosinus
     fort              0.9136
 vraiment              0.9216
   rapide              0.9361
olympique              0.9445
   sprint              0.9519
   course              0.9570
      ski              0.9649
  coucher              0.9711
      tir              0.9768
 médaille              0.9789
 secondes              0.9832


We apply the same logic to the gender of the commentators.

In [21]:
STOPWORDS = [x.strip() for x in open('stopwords.txt').readlines()]

words = " ".join(df["text"].tolist()).split()
words_filtered = [w for w in words if w not in STOPWORDS and len(w) > 2]
freq = Counter(words_filtered).most_common(50)

words_H = " ".join(df[df["main_g"] == "male"]["text"].tolist()).split()
words_F = " ".join(df[df["main_g"] == "female"]["text"].tolist()).split()

counter_H = Counter([w for w in words_H if w not in STOPWORDS and len(w) > 2])
counter_F = Counter([w for w in words_F if w not in STOPWORDS and len(w) > 2])

print("=== 50 mots les plus fréquents (hors stopwords) ===")
for word, count in freq:
    h = counter_H.get(word, 0)
    f = counter_F.get(word, 0)
    total = h + f
    pct_h = h / total * 100 if total > 0 else 0
    pct_f = f / total * 100 if total > 0 else 0
    print(f"  {word}: {count} (male: {pct_h:.0f}% | female: {pct_f:.0f}%)")
print(f"Total male : {len([w for w in words_H if w not in STOPWORDS and len(w) > 2])/len(words_filtered)} | female : {len([w for w in words_F if w not in STOPWORDS and len(w) > 2])/len(words_filtered)}")

=== 50 mots les plus fréquents (hors stopwords) ===
  'est: 682 (male: 74% | female: 26%)
  secondes: 671 (male: 88% | female: 12%)
  tir: 456 (male: 72% | female: 28%)
  course: 306 (male: 69% | female: 31%)
  relais: 277 (male: 88% | female: 12%)
  faut: 247 (male: 66% | female: 34%)
  temps: 246 (male: 78% | female: 22%)
  médaille: 237 (male: 92% | female: 8%)
  vraiment: 234 (male: 66% | female: 34%)
  tête: 230 (male: 85% | female: 15%)
  petit: 227 (male: 75% | female: 25%)
  déjà: 215 (male: 88% | female: 12%)
  skis: 190 (male: 78% | female: 22%)
  également: 183 (male: 93% | female: 7%)
  vite: 182 (male: 82% | female: 18%)
  français: 176 (male: 93% | female: 7%)
  piste: 173 (male: 80% | female: 20%)
  peutêtre: 169 (male: 83% | female: 17%)
  monde: 167 (male: 93% | female: 7%)
  l'instant: 155 (male: 90% | female: 10%)
  voit: 147 (male: 75% | female: 25%)
  chercher: 145 (male: 86% | female: 14%)
  qu'on: 145 (male: 73% | female: 27%)
  olympique: 143 (male: 93% | female

In [22]:
embeddings_path = data_path+"embeddings.npy"

if os.path.exists(embeddings_path):
    print("\nChargement des embeddings depuis le cache...")
    embeddings = np.load(embeddings_path)
    print(f"Embeddings chargés : {embeddings.shape}")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nCalcul des embeddings sur : {device}")

    embedding_model = SentenceTransformer("dangvantuan/sentence-camembert-base", device=device)

    embeddings = embedding_model.encode(
        df["text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    np.save(embeddings_path, embeddings)
    print(f"Embeddings sauvegardés dans {embeddings_path}")

df["embedding"] = list(embeddings)
print(f"Corpus : {len(df)} phrases | embeddings : {embeddings.shape}")



Chargement des embeddings depuis le cache...
Embeddings chargés : (8844, 768)
Corpus : 8844 phrases | embeddings : (8844, 768)


In [24]:
TERMES = ["tir", "course", "sprint", "olympique", "secondes", "coucher", "ski", "vraiment","médaille","fort","rapide"]

results = []

for terme in TERMES:
    mask_F = df["main_g"] == "female"
    mask_H = df["main_g"] == "male"
    mask_terme = df["text"].str.contains(terme, case=False, na=False)

    emb_F = np.array(df[mask_F & mask_terme]["embedding"].tolist())
    emb_H = np.array(df[mask_H & mask_terme]["embedding"].tolist())

    if len(emb_F) == 0 or len(emb_H) == 0:
        results.append({"terme": terme,  "similarite_cosinus": None})
        continue

    centroid_F = emb_F.mean(axis=0, keepdims=True)
    centroid_H = emb_H.mean(axis=0, keepdims=True)

    sim = cosine_similarity(centroid_F, centroid_H)[0][0]
    results.append({"terme": terme, "similarite_cosinus": round(sim, 4)})

results_df = pd.DataFrame(results).dropna(subset=["similarite_cosinus"]).sort_values("similarite_cosinus")
print("\n=== Similarité cosinus F vs H par terme ===")
print(results_df.to_string(index=False))


=== Similarité cosinus F vs H par terme ===
    terme  similarite_cosinus
  coucher              0.8712
olympique              0.9097
   rapide              0.9157
   sprint              0.9291
     fort              0.9471
   course              0.9597
 vraiment              0.9603
 médaille              0.9662
      tir              0.9714
      ski              0.9716
 secondes              0.9786
